# Export GeoTIFF rasters

This recipe saves **pixels instead of zonal statistics**, using the local backend without a Google Earth Engine account. It exports:

1. Quality-filtered, scaled, crop-masked MODIS EVI: one TIFF per observation.
2. Esri/Impact Observatory land-cover classes at 10 m resolution.
3. A binary cropland mask at 10 m resolution.

The example uses a small area in Kenya to keep downloads and memory use modest. Internet access is required. MODIS observations are already 16-day satellite composites; `load_evi()` adds no temporal or spatial aggregation. Land cover is resampled by mode to the MODIS grid **only when used to mask EVI**.

See the [API reference](../../docs/api-reference.md#raster-loading-and-geotiff-export) for the available options.

In [1]:
from pathlib import Path
import tempfile

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from shapely.geometry import box

import evy

# A small bounding box in Kenya; replace this with your own GeoDataFrame.
boundaries = gpd.GeoDataFrame(
    geometry=[box(36.01, -1.01, 36.02, -1.00)], crs="EPSG:4326"
)
start_date = "2023-01-01"
end_date = "2023-01-31"
landcover_year = 2025

# A new folder makes rerunning safe: export_geotiff never overwrites files.
# Replace this with Path("your/output/folder") for a persistent destination.
output_dir = Path(tempfile.mkdtemp(prefix="evy-geotiff-"))

/Users/farhanreynaldo/Documents/world-bank/git-repo/evy/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Export EVI observations

`load_evi()` applies quality filtering, the MODIS EVI scale factor, and the selected crop mask. With `landcover_source="esri"`, every observation uses its year's land-cover map (2023 here). The supported Esri years are 2017–2025; unavailable years raise an error.

EVI stays on the original MODIS grid, at nominal 250 m resolution. It does not become 10 m EVI. Set `mask_cropland=False` to export quality-filtered EVI across all land-cover classes.

In [2]:
evi = evy.load_evi(
    boundaries,
    start_date,
    end_date,
    provider="planetary_computer",
    landcover_source="esri",
)
evi_paths = evy.export_geotiff(evi, output_dir / "evi")
print(f"Exported {len(evi_paths)} EVI observations:")
for path in evi_paths:
    print(path.name)

Exported 4 EVI observations:
evi_2023-01-01.tif
evi_2023-01-09.tif
evi_2023-01-17.tif
evi_2023-01-25.tif


## Export native land-cover classes

Without an EVI grid argument, `load_landcover()` loads Esri/Impact Observatory data at 10 m resolution. This example exports the **2025** map independently of the 2023 EVI observations above. Classes include **5 = crops**, **2 = trees**, and **0 = nodata**.

The source is the public v003 download assets linked from the [Esri Land Cover Explorer](https://livingatlas.arcgis.com/landcoverexplorer/).

In [3]:
landcover = evy.load_landcover(boundaries, source="esri", year=landcover_year)
landcover_paths = evy.export_geotiff(
    landcover.rename(f"landcover_{landcover_year}"), output_dir / "landcover"
)
print(landcover_paths[0].name)

landcover_2025.tif


## Export a binary cropland mask

Convert the land-cover map to **1 = cropland**, **0 = other classes**, and **255 = nodata**. Keeping nodata distinct from zero prevents missing coverage from being treated as non-cropland.

In [4]:
cropland = (landcover == 5).where(landcover != 0, 255).astype("uint8")
cropland = cropland.rename(f"cropland_{landcover_year}").rio.write_nodata(255)
cropland_paths = evy.export_geotiff(cropland, output_dir / "cropland")
print(cropland_paths[0].name)

cropland_2025.tif


## Reopen the exported files

Use Rasterio to inspect the files' dimensions, resolution, and nodata. The checks below also compare a reopened EVI raster and crop mask with the arrays used to write them. An EVI image can contain nodata where quality is poor or pixels are not classified as crops.

In [5]:
rows = []
for path in [*evi_paths, *landcover_paths, *cropland_paths]:
    with rasterio.open(path) as src:
        rows.append(
            {
                "file": path.name,
                "width": src.width,
                "height": src.height,
                "pixel_size": tuple(round(value, 3) for value in src.res),
                "nodata": src.nodata,
            }
        )

with rasterio.open(evi_paths[0]) as src:
    assert src.crs == evi.rio.crs
    np.testing.assert_allclose(src.read(1), evi.isel(time=0).values, equal_nan=True)
with rasterio.open(landcover_paths[0]) as src:
    assert src.res == (10.0, 10.0)
    assert src.nodata == 0
with rasterio.open(cropland_paths[0]) as src:
    assert src.nodata == 255
    assert set(np.unique(src.read(1))) <= {0, 1, 255}
    np.testing.assert_array_equal(src.read(1), cropland.values)

pd.DataFrame(rows)

,file,width,height,pixel_size,nodata
0,evi_2023-01-01.tif,5,5,"(231.656, 231.656)",NaN
1,evi_2023-01-09.tif,5,5,"(231.656, 231.656)",NaN
2,evi_2023-01-17.tif,5,5,"(231.656, 231.656)",NaN
3,evi_2023-01-25.tif,5,5,"(231.656, 231.656)",NaN
4,landcover_2025.tif,113,111,"(10.0, 10.0)",0.0
5,cropland_2025.tif,113,111,"(10.0, 10.0)",255.0


The files are saved under `output_dir`; open them in QGIS or another raster viewer. The default temporary directory may eventually be cleaned by the operating system, so choose a persistent destination for work you want to keep.

Exports cover the input boundaries' **bounding box**, including any pixels outside the polygons. They preserve the raster CRS and nodata, and existing files are never overwritten. Split large regions or long date ranges to manage memory and download size.